# Fine-tuning Flan-T5-base for Indian criminal law after the 2024 recodification## 1. Project summaryOn **1 July 2024** India replaced three colonial-era criminal codes at once. TheIndian Penal Code, 1860 became the **Bharatiya Nyaya Sanhita, 2023** (358sections), the Code of Criminal Procedure, 1973 became the **Bharatiya NagarikSuraksha Sanhita, 2023** (531 sections), and the Indian Evidence Act, 1872 becamethe **Bharatiya Sakshya Adhiniyam, 2023** (170 sections). Every section wasrenumbered, many were merged or split, and some were repealed outright.That creates a problem general-purpose chatbots handle badly. Their training datastraddles the changeover, so they answer "which section covers cheating?" withIPC 420 — repealed — rather than BNS 318, and they do it confidently. Worse, somenumbers survive into the new code attached to an entirely different provision:CrPC 482 was the High Court's inherent power to quash proceedings, while **BNSS482 is anticipatory bail**. A model working from parametric memory has no way tonotice it has answered the wrong question.This project builds an assistant that answers from authoritative retrieved textinstead. This notebook is **Phase 2**: fine-tuning `google/flan-t5-base` on14,524 instruction/QA pairs built from the official Ministry of Home Affairsgazette PDFs, the Bureau of Police Research and Development correspondencetables, the BNSS First Schedule, and 174 Supreme Court judgments. Phase 3 addsthe FAISS retrieval layer on top of this model.**Scope discipline.** The bot describes what the law says and cites its source.It does not give legal advice, does not suggest ways to evade liability, and doesnot claim to represent an advocate. The training data contains a dedicated`scope` family teaching exactly that boundary.**Industry:** Government and Public Administration.**GitHub repository:** https://github.com/suryanshu-g/legal-llm-bot**Data provenance and known limitations:** see `data/DATA_REPORT.md` in the repo.

---## 2. SetupTwo things to set before running: the repository URL, and (optionally) a HuggingFace token if you want the trained model pushed to the Hub.**Runtime:** Runtime → Change runtime type → **T4 GPU**. Training on CPU is notpractical here.

In [ ]:
# ----------------------------------------------------------------- settings# The public GitHub repository holding the datasets (Phase 1 / 1.5 output).REPO_URL = "https://github.com/suryanshu-g/legal-llm-bot.git"# Where the fine-tuned model is pushed, if a Hugging Face token is available.# Leave as None to fall back to saving on Google Drive.HF_REPO_ID = "legal-llm-bot-flan-t5-base"SEED = 20240701MODEL_NAME = "google/flan-t5-base"

In [ ]:
import subprocess, sysprint(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or      "No GPU detected - set Runtime > Change runtime type > T4 GPU")

In [ ]:
%pip install -q "transformers>=4.41" "datasets>=2.19" "accelerate>=0.30" \                "evaluate>=0.4" sentencepiece

In [ ]:
import os, json, re, time, random, textwrapfrom collections import Counter, defaultdictimport numpy as npimport torchimport matplotlib.pyplot as pltimport transformersfrom transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,                          DataCollatorForSeq2Seq, Seq2SeqTrainer,                          Seq2SeqTrainingArguments, EarlyStoppingCallback,                          set_seed)from datasets import Datasetset_seed(SEED)random.seed(SEED); np.random.seed(SEED)DEVICE = "cuda" if torch.cuda.is_available() else "cpu"print("transformers", transformers.__version__, "| torch", torch.__version__,      "| device", DEVICE)if DEVICE == "cuda":    print("GPU:", torch.cuda.get_device_name(0),          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone the repo so the data files come straight from source control rather# than being uploaded by hand - the notebook is then reproducible from a fresh# runtime with nothing but this URL.REPO_DIR = "/content/legal-llm-bot"if REPO_URL.startswith("REPO_URL"):    raise SystemExit("Set REPO_URL in the settings cell above before running.")if not os.path.isdir(REPO_DIR):    !git clone --depth 1 {REPO_URL} {REPO_DIR}else:    print("already cloned")DATA = os.path.join(REPO_DIR, "data", "processed")print(sorted(f for f in os.listdir(DATA) if f.endswith((".jsonl", ".csv"))))

---## 3. Data loading and industry relevanceFour files matter here:| File | Rows | Role ||---|---|---|| `train.jsonl` | 11,751 | training || `val.jsonl` | 1,369 | early stopping and model selection || `test.jsonl` | 676 | final evaluation, touched once || `confusion_test_set.jsonl` | 44 | held out of all three, for the Phase 4 comparison |Each split has an aligned `*_index.jsonl` giving the `qa_type` of every row, sometrics can be broken down by question type rather than reported as a singlenumber that hides where the model is actually weak.**The splits are leakage-safe at the level of facts, not rows.** Every fact inthis dataset is asked several ways — "What does BNS 103 say?" and "Explain BNSSection 103." are one fact in two wordings — so a random row split would put onein train and its paraphrase in validation, and the validation score wouldpartly measure memorised phrasing. Instead whole *groups* are split, where agroup is a section, a judgment, or a date, and groups are further merged acrossthe old-to-new concordance so that "which BNS section replaced IPC 302?" and"which IPC section does BNS 103 come from?" cannot land on opposite sides. Thatsecond step matters most: the old-to-new correspondence *is* the project'sclaim, so letting it straddle the split would flatter exactly the capabilitybeing tested.

In [ ]:
def load_jsonl(path):    with open(path, encoding="utf-8") as fh:        return [json.loads(line) for line in fh if line.strip()]splits, index = {}, {}for name in ("train", "val", "test"):    splits[name] = load_jsonl(os.path.join(DATA, f"{name}.jsonl"))    index[name] = load_jsonl(os.path.join(DATA, f"{name}_index.jsonl"))    assert len(splits[name]) == len(index[name]), f"{name}: split/index misaligned"confusion = load_jsonl(os.path.join(DATA, "confusion_test_set.jsonl"))total = sum(len(v) for v in splits.values())for name, rows in splits.items():    print(f"{name:<6} {len(rows):>6} rows  ({100 * len(rows) / total:.1f}%)")print(f"{'conf':<6} {len(confusion):>6} rows  (held out of all three)")

In [ ]:
# Question-type mix per split. These should track each other closely; the# splitter assigns each group to whichever split is least ahead of its quota# for the question types that group contains.types = sorted({m["qa_type"] for m in index["train"]})print(f"{'qa_type':<24}" + "".join(f"{n:>12}" for n in ("train", "val", "test")))for t in types:    row = f"{t:<24}"    for name in ("train", "val", "test"):        c = sum(1 for m in index[name] if m["qa_type"] == t)        row += f"{c:>6} {100 * c / len(index[name]):>4.1f}%"    print(row)

In [ ]:
# A few rows, one per question type, to show what the model is being asked.seen = set()for row, meta in zip(splits["train"], index["train"]):    if meta["qa_type"] in seen:        continue    seen.add(meta["qa_type"])    print(f"[{meta['qa_type']}]")    print("  Q:", row["instruction"])    print("  A:", textwrap.shorten(row["output"], 240, placeholder=" ..."))    print()

### Why this dataset reflects the chosen industryThe submission industry is **Government and Public Administration**, and the datais drawn almost entirely from the machinery of the state rather than fromcommentary about it:* **The statutory text** is the Ministry of Home Affairs gazette PDFs of the  three Acts as published — the operative law itself, 100% of all 1,059 sections  across BNS, BNSS and BSA.* **The old-to-new concordance** comes from the "Correspondence Table and  Comparison Summary" PDFs published by the **Bureau of Police Research and  Development**, an MHA body, cross-checked against the **Uttar Pradesh Police**  comparative table. These are the documents police forces themselves were given  to work through the transition.* **The offence classification** — cognizable or not, bailable or not, which  court may try it — is the **BNSS First Schedule**, which is what determines  whether a police officer may arrest without a warrant. That is administrative  procedure, not legal theory.* **The case law** is 174 Supreme Court judgments, the public record of how the  state's courts have construed these provisions.The users this serves are the ones administering the system: police recording anFIR under the right section, prosecutors framing charges, court staff, andcitizens trying to understand a provision cited at them. The 1 July 2024changeover is a live administrative problem — an offence on 30 June is chargedunder the IPC and one on 1 July under the BNS — and getting the section numberwrong has real consequences for a real proceeding.

---## 4. Preprocessing### The prompt templateFlan-T5 was instruction-tuned on a large mixture where each example is a plainnatural-language instruction with a short task-describing prefix. Staying closeto that format means the fine-tune adapts a behaviour the model already hasrather than teaching it a new input convention from scratch. The template is:```answer the indian criminal law question: {instruction}```and when a row carries a non-empty `input` field, that is appended as context:```answer the indian criminal law question: {instruction}context: {input}```Every row in the current dataset has an empty `input`, but the schema allows itand **Phase 3 will use exactly that slot to inject retrieved passages**. Trainingwith the template already in place means the retrieval layer can be addedwithout changing the input format the model was tuned on.The prefix is lowercase and unpunctuated to match Flan's own task prefixes.Targets are the `output` field unchanged.### Sequence lengthsSet from the data rather than guessed. Measured over the training split withthis tokenizer:| | p50 | p95 | p99 | max ||---|---|---|---|---|| source tokens | 23 | 38 | 47 | 72 || target tokens | 59 | 234 | 265 | 308 |So `max_source_length = 96` and `max_target_length = 320` truncate **nothing** —verified in the cell below rather than asserted. Padding is dynamic, done perbatch by `DataCollatorForSeq2Seq`, so the short sources cost nothing.Label padding is set to `-100` so pad positions are ignored by the loss.

In [ ]:
TASK_PREFIX = "answer the indian criminal law question: "MAX_SOURCE_LENGTH = 96MAX_TARGET_LENGTH = 320tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)def build_prompt(row):    """The input string the model sees. Phase 3 fills `input` with retrieval."""    prompt = TASK_PREFIX + row["instruction"].strip()    context = (row.get("input") or "").strip()    if context:        prompt += "\n\ncontext: " + context    return promptprint(build_prompt(splits["train"][0]))print("---")print(splits["train"][0]["output"][:200])

In [ ]:
# Confirm the chosen limits truncate nothing before committing to them.def length_report(rows, name):    src = [len(tokenizer(build_prompt(r)).input_ids) for r in rows]    tgt = [len(tokenizer(r["output"]).input_ids) for r in rows]    over_s = sum(1 for x in src if x > MAX_SOURCE_LENGTH)    over_t = sum(1 for x in tgt if x > MAX_TARGET_LENGTH)    print(f"{name:<6} source max {max(src):>4} (> {MAX_SOURCE_LENGTH}: {over_s})"          f"   target max {max(tgt):>4} (> {MAX_TARGET_LENGTH}: {over_t})")    return over_s + over_ttruncated = sum(length_report(splits[n], n) for n in ("train", "val", "test"))print("\nrows that would be truncated:", truncated)

In [ ]:
def tokenize(batch):    model_inputs = tokenizer(batch["prompt"], max_length=MAX_SOURCE_LENGTH,                             truncation=True)    labels = tokenizer(text_target=batch["output"], max_length=MAX_TARGET_LENGTH,                       truncation=True)    model_inputs["labels"] = labels["input_ids"]    return model_inputsdef to_dataset(rows):    ds = Dataset.from_list([{"prompt": build_prompt(r), "output": r["output"]}                            for r in rows])    return ds.map(tokenize, batched=True, remove_columns=ds.column_names,                  desc="tokenizing")tokenized = {n: to_dataset(splits[n]) for n in ("train", "val")}print(tokenized["train"])

---## 5. Model and fine-tuning### Why Flan-T5-base`google/flan-t5-base` is ~250M parameters: small enough to fine-tune fully on asingle T4 in a reasonable time, and already instruction-tuned, so it starts from"follow the instruction" rather than "continue the text". A decoder-only model ofsimilar size would need more data to reach the same instruction-followingbehaviour, and the encoder-decoder shape suits Phase 3 — retrieved passages go inthe encoder where they can be attended over without competing with the answer fordecoder context.### Hyperparameters, and why| Setting | Value | Reasoning ||---|---|---|| Learning rate | `3e-4` | T5 is normally fine-tuned at 1e-4 to 5e-4 with AdamW. 3e-4 sits mid-band: fast enough to converge in a few epochs on 11.7k examples, below the point where T5 fine-tuning tends to go unstable. || Scheduler | linear, 5% warmup | Warmup matters for T5, whose layer norms are sensitive to a cold start at this learning rate. || Optimiser | AdamW, `weight_decay=0.01` | Standard; mild decay for regularisation on a dataset with heavily repeated structure. || Batch size | 8 × 2 accumulation = 16 effective | 8 fits a T4 comfortably at these lengths. Accumulation gives a steadier gradient without more memory. || Epochs | ≤ 25, early stopping patience 3 | See below. || Precision | **fp32** | See below — this one is a trap. || Model selection | lowest `eval_loss`, `load_best_model_at_end` | The final epoch is usually not the best one. |### Do not use fp16 hereT5 and its derivatives are known to produce `NaN` losses under fp16 becauseactivations in the attention blocks overflow the fp16 range. The usual fix isbf16, which has the range but not the precision problem — except **the T4 is aTuring card (sm_75) and has no bf16 support**; bf16 needs Ampere (sm_80) ornewer. So on a T4 the only safe choice is fp32. It is slower, and it is correct.The cell below detects the GPU and enables bf16 only if the hardware actuallysupports it, so the notebook does the right thing on an A100 too.### Early stopping, not 25 blind epochs25 epochs over 11,751 examples is roughly 18,000 optimiser steps. On a datasetthis templated the model will fit the phrasing long before that and then startmemorising. Validation loss is evaluated every epoch and training stops after 3evaluations without improvement, keeping the best checkpoint. **25 is a ceiling,not a target** — expect it to stop well short.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)print(f"{model.num_parameters() / 1e6:.1f}M parameters")# bf16 only where the hardware supports it; never fp16 for T5 (NaN losses).USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()print("bf16:", USE_BF16, "| fp16: False (T5 overflows in fp16)")

In [ ]:
OUTPUT_DIR = "/content/outputs/flan-t5-base-legal"BATCH_SIZE = 8GRAD_ACCUM = 2args = Seq2SeqTrainingArguments(    output_dir=OUTPUT_DIR,    seed=SEED,    num_train_epochs=25,                 # hard ceiling; early stopping decides    learning_rate=3e-4,    lr_scheduler_type="linear",    warmup_ratio=0.05,    weight_decay=0.01,    optim="adamw_torch",    per_device_train_batch_size=BATCH_SIZE,    per_device_eval_batch_size=BATCH_SIZE * 2,    gradient_accumulation_steps=GRAD_ACCUM,    bf16=USE_BF16,    fp16=False,    eval_strategy="epoch",    save_strategy="epoch",    save_total_limit=2,    logging_strategy="epoch",    load_best_model_at_end=True,    metric_for_best_model="eval_loss",    greater_is_better=False,    predict_with_generate=False,         # generation is done separately, batched    report_to="none",)collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)trainer = Seq2SeqTrainer(    model=model,    args=args,    train_dataset=tokenized["train"],    eval_dataset=tokenized["val"],    data_collator=collator,    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],)print("effective batch size:", BATCH_SIZE * GRAD_ACCUM)

In [ ]:
t0 = time.time()train_result = trainer.train()TRAIN_MINUTES = (time.time() - t0) / 60print(f"\ntrained for {TRAIN_MINUTES:.1f} minutes")print(train_result.metrics)

In [ ]:
# Loss curve. Two things to look for: validation loss flattening (the model has# learnt what it can) and validation loss turning up while training loss keeps# falling (overfitting - which is what early stopping is there to catch).hist = trainer.state.log_historytr = [(h["epoch"], h["loss"]) for h in hist if "loss" in h]ev = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]plt.figure(figsize=(8, 4.5))plt.plot(*zip(*tr), marker="o", label="training loss")plt.plot(*zip(*ev), marker="s", label="validation loss")if ev:    best = min(ev, key=lambda p: p[1])    plt.axvline(best[0], ls="--", c="gray", lw=1)    plt.annotate(f"best: {best[1]:.4f} @ epoch {best[0]:.0f}",                 xy=best, xytext=(6, 12), textcoords="offset points")plt.xlabel("epoch"); plt.ylabel("cross-entropy loss")plt.title("Fine-tuning flan-t5-base on Indian criminal law QA")plt.legend(); plt.grid(alpha=.3); plt.tight_layout()plt.savefig("/content/loss_curve.png", dpi=150)plt.show()EPOCHS_RUN = max(e for e, _ in ev) if ev else 0FINAL_TRAIN_LOSS = tr[-1][1] if tr else float("nan")BEST_VAL_LOSS = min(v for _, v in ev) if ev else float("nan")print(f"epochs run: {EPOCHS_RUN:.0f}/25 | final train loss {FINAL_TRAIN_LOSS:.4f}"      f" | best val loss {BEST_VAL_LOSS:.4f}")

---## 6. Evaluation### Which metrics, and why three of them**Exact match** and **token-level F1** are the SQuAD-style pair: both normaliseby lowercasing, stripping punctuation and dropping articles. They are reportedbecause they are standard and legible.But exact match is a harsh metric for this data. The answers are full sentences —median 59 tokens — not short spans, so a reply that names the right section andstates the right rule in slightly different words scores zero. Read F1 as themain general-quality number and exact match as a floor.Neither, though, measures the thing this project actually claims. So there is athird:**Citation accuracy.** Extract every statutory reference (`BNS 103`, `IPC 302`,`BNSS 482`, ...) from the gold answer and from the prediction, and ask whetherthe model named the right provisions. This is the metric that speaks to thedifferentiator thesis: on an `old_to_new` question, naming BNS 103 for IPC 302 isthe whole job, and getting the surrounding prose slightly different does notmatter. It is reported as precision, recall and F1 over reference sets, plus astrict "all gold references present" accuracy.### Broken down by question typeAn aggregate would hide the only number that matters. `section_text` is a thirdof the test set and is essentially recall of memorised statute text, so a strongaggregate could be carried entirely by that while the mapping questions —`old_to_new` and `new_to_old`, the project's actual claim — perform badly. Everymetric below is therefore reported per `qa_type` as well as overall.

In [ ]:
ARTICLES = re.compile(r"\b(a|an|the)\b")PUNCT = re.compile(r"[^\w\s]")def normalise(s):    """SQuAD-style normalisation: case, punctuation and articles removed."""    s = PUNCT.sub(" ", s.lower())    return " ".join(ARTICLES.sub(" ", s).split())def exact_match(pred, gold):    return float(normalise(pred) == normalise(gold))def token_f1(pred, gold):    p, g = normalise(pred).split(), normalise(gold).split()    if not p or not g:        return float(p == g)    common = Counter(p) & Counter(g)    overlap = sum(common.values())    if overlap == 0:        return 0.0    precision, recall = overlap / len(p), overlap / len(g)    return 2 * precision * recall / (precision + recall)

In [ ]:
# ---- statutory reference extraction ---------------------------------------# The dataset writes citations several ways - "BNS Section 103", "BNS 103",# "Section 477 of the Code of Criminal Procedure, 1973", and enumerations like# "IPC Sections 415, 417, 418, 419 and 420" - so all of those have to parse.ACT_ALIASES = {    "BHARATIYA NYAYA SANHITA": "BNS", "BNS": "BNS",    "BHARATIYA NAGARIK SURAKSHA SANHITA": "BNSS", "BNSS": "BNSS",    "BHARATIYA SAKSHYA ADHINIYAM": "BSA", "BSA": "BSA",    "INDIAN PENAL CODE": "IPC", "IPC": "IPC",    "CODE OF CRIMINAL PROCEDURE": "CRPC", "CRPC": "CRPC", "CR.P.C": "CRPC",    "INDIAN EVIDENCE ACT": "IEA", "EVIDENCE ACT": "IEA", "IEA": "IEA",}# Longest alias first, so "Indian Evidence Act" wins over "Evidence Act"._ACTS = "|".join(re.escape(a) for a in sorted(ACT_ALIASES, key=len, reverse=True))_NUMS = r"\d+[A-Za-z]{0,2}(?:\s*(?:,|and|&)\s*\d+[A-Za-z]{0,2})*"# The optional \d{4} skips the year in "the Indian Penal Code, 1860" so that a# year is never mistaken for a section number.REF_ACT_FIRST = re.compile(    r"(" + _ACTS + r")\b[ ,]*(?:\d{4}[ ,]*)?(?:Sections?|ss?\.)?\s*(" + _NUMS + r")",    re.I)REF_SEC_FIRST = re.compile(    r"Sections?\s*(" + _NUMS + r")\s*(?:of\s+(?:the\s+)?)(" + _ACTS + r")\b", re.I)def _numbers(blob):    for part in re.split(r"\s*(?:,|and|&)\s*", blob):        m = re.fullmatch(r"(\d+)([A-Z]{0,2})", part.strip().upper())        if m and int(m.group(1)) < 1000:      # >= 1000 is a year, not a section            yield m.group(1) + m.group(2)def extract_refs(text):    """The set of statutory references a passage cites, e.g. {'BNS 103'}."""    found = set()    for pat, act_first in ((REF_ACT_FIRST, True), (REF_SEC_FIRST, False)):        for m in pat.finditer(text):            act = (m.group(1) if act_first else m.group(2)).upper().rstrip(".")            nums = m.group(2) if act_first else m.group(1)            canon = ACT_ALIASES.get(act)            if canon:                found.update(f"{canon} {n}" for n in _numbers(nums))    return found# Verify against the forms that actually occur before trusting the metric.checks = [    ("IPC Section 302 corresponds to BNS Section 103.", {"IPC 302", "BNS 103"}),    ("CrPC 438 is now BNSS 482.", {"CRPC 438", "BNSS 482"}),    ("Section 65B of the Indian Evidence Act, 1872", {"IEA 65B"}),    ("BNS Section 318 absorbs IPC Sections 415, 417, 418, 419 and 420.",     {"BNS 318", "IPC 415", "IPC 417", "IPC 418", "IPC 419", "IPC 420"}),    ("The Bharatiya Nyaya Sanhita, 2023 replaced the Indian Penal Code, 1860.",     set()),]for text, want in checks:    got = extract_refs(text)    assert got == want, f"{text!r}\n  got  {sorted(got)}\n  want {sorted(want)}"print(f"reference extractor: {len(checks)}/{len(checks)} checks pass")# Why this metric earns its place: token F1 hardly notices a wrong section.gold = "IPC Section 302 corresponds to BNS Section 103 (Punishment for murder)."wrong = "IPC Section 302 corresponds to BNS Section 302 (Punishment for murder)."print(f"  a wrong-section answer scores token F1 {token_f1(wrong, gold):.3f} "      f"but fails the citation check ({extract_refs(gold) <= extract_refs(wrong)})")

In [ ]:
@torch.no_grad()def generate(rows, batch_size=32, max_new_tokens=MAX_TARGET_LENGTH):    """Greedy decoding over a list of rows; returns the predicted strings."""    model.eval()    preds = []    for start in range(0, len(rows), batch_size):        chunk = rows[start:start + batch_size]        enc = tokenizer([build_prompt(r) for r in chunk], return_tensors="pt",                        padding=True, truncation=True,                        max_length=MAX_SOURCE_LENGTH).to(model.device)        out = model.generate(**enc, max_new_tokens=max_new_tokens, num_beams=1)        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))        print(f"\rgenerated {min(start + batch_size, len(rows))}/{len(rows)}",              end="")    print()    return predst0 = time.time()test_preds = generate(splits["test"])print(f"test-set generation took {(time.time() - t0) / 60:.1f} min")

In [ ]:
def score(rows, preds, metas=None):    """Per-row metrics, plus the aggregate and the per-qa_type breakdown."""    per_row = []    for i, (row, pred) in enumerate(zip(rows, preds)):        gold = row["output"]        gold_refs, pred_refs = extract_refs(gold), extract_refs(pred)        hits = len(gold_refs & pred_refs)        per_row.append({            "qa_type": metas[i]["qa_type"] if metas else "all",            "em": exact_match(pred, gold),            "f1": token_f1(pred, gold),            "cite_p": hits / len(pred_refs) if pred_refs else (1.0 if not gold_refs else 0.0),            "cite_r": hits / len(gold_refs) if gold_refs else 1.0,            "cite_all": float(gold_refs <= pred_refs),            "has_refs": bool(gold_refs),        })    return per_rowdef summarise(per_row, label):    def agg(rows_, key):        vals = [r[key] for r in rows_]        return 100 * sum(vals) / len(vals) if vals else float("nan")    # Some question types (transition, scope) have no statutory citation in the    # gold answer at all, so the citation metrics are simply not defined there    # and are reported as None rather than as a misleading zero.    cited = [r for r in per_row if r["has_refs"]]    if cited:        cp, cr = agg(cited, "cite_p"), agg(cited, "cite_r")        cf1 = 2 * cp * cr / (cp + cr) if (cp + cr) else 0.0        all_cites = agg(cited, "cite_all")    else:        cf1 = all_cites = None    return {        "set": label, "n": len(per_row),        "exact_match": agg(per_row, "em"),        "f1": agg(per_row, "f1"),        "citation_f1": cf1,        "all_citations_present": all_cites,        "n_with_citations": len(cited),    }def pct(v):    """Format a metric that may be undefined for this slice."""    return "     n/a" if v is None else f"{v:>7.1f}%"test_rows = score(splits["test"], test_preds, index["test"])overall = summarise(test_rows, "test (overall)")print(f"{'set':<26}{'n':>6}{'EM':>9}{'F1':>9}{'citeF1':>9}{'allCites':>10}")print("-" * 69)print(f"{overall['set']:<26}{overall['n']:>6}{pct(overall['exact_match'])}"      f"{pct(overall['f1'])}{pct(overall['citation_f1'])}"      f"{pct(overall['all_citations_present'])}")by_type = []for t in sorted({r["qa_type"] for r in test_rows}):    sub = [r for r in test_rows if r["qa_type"] == t]    s = summarise(sub, t)    by_type.append(s)    print(f"{'  ' + t:<26}{s['n']:>6}{pct(s['exact_match'])}{pct(s['f1'])}"          f"{pct(s['citation_f1'])}{pct(s['all_citations_present'])}")

In [ ]:
# Same numbers as a chart, since the per-type spread is the interesting part.labels = [s["set"] for s in by_type]x = np.arange(len(labels))plt.figure(figsize=(11, 4.5))for i, (key, name) in enumerate([("f1", "token F1"),                                 ("citation_f1", "citation F1"),                                 ("exact_match", "exact match")]):    plt.bar(x + (i - 1) * 0.27, [s[key] or 0 for s in by_type], width=0.27,            label=name)plt.xticks(x, labels, rotation=30, ha="right")plt.ylabel("%"); plt.ylim(0, 100); plt.legend(); plt.grid(axis="y", alpha=.3)plt.title("Test-set performance by question type")plt.tight_layout(); plt.savefig("/content/metrics_by_qa_type.png", dpi=150)plt.show()

In [ ]:
# A look at where it goes wrong, which is more informative than the mean.worst = sorted(range(len(test_rows)), key=lambda i: test_rows[i]["f1"])[:5]for i in worst:    print(f"[{test_rows[i]['qa_type']}]  F1 {test_rows[i]['f1']:.2f}")    print("  Q   :", splits["test"][i]["instruction"])    print("  gold:", textwrap.shorten(splits["test"][i]["output"], 200, placeholder=" ..."))    print("  pred:", textwrap.shorten(test_preds[i], 200, placeholder=" ..."))    print()

---## 7. The confusion test set44 questions held out of train, validation and test alike, each one aimed at aspecific way the renumbering traps a model working from memory:| Kind | Entries | The trap ||---|---|---|| `collision` | 14 | The number survives but attaches to an unrelated provision — CrPC 482 was inherent powers, BNSS 482 is anticipatory bail || `merged` | 14 | Several old sections folded into one — BNS 318 absorbs IPC 415, 417, 418, 419 and 420 || `split` | 5 | One old section broken across several — IPC 498A became BNS 85 *and* 86 || `removed` | 10 | Repealed outright — any section number in reply is wrong by construction || `transition` | 1 | Which code applies turns on the date of the offence, not the date of the trial |**Expect this to be the weakest result in the notebook, and that is the point.**This is a fine-tuned model with no retrieval. It has seen the concordance duringtraining, but these particular facts were deliberately withheld, so it can onlyanswer from what it generalised. Phase 3 adds retrieval over`retrieval_corpus.jsonl`, where each chunk carries its old/new counterpart in theembedded text, and Phase 4 re-runs this exact set against both this model and ageneral-purpose LLM. The number below is the **baseline** that comparison ismeasured against — record it.

In [ ]:
conf_preds = generate(confusion)conf_rows = score(confusion, conf_preds,                  [{"qa_type": e["mapping_type"]} for e in confusion])conf_overall = summarise(conf_rows, "confusion (overall)")print(f"{'kind':<26}{'n':>6}{'EM':>9}{'F1':>9}{'citeF1':>9}{'allCites':>10}")print("-" * 69)print(f"{conf_overall['set']:<26}{conf_overall['n']:>6}"      f"{pct(conf_overall['exact_match'])}{pct(conf_overall['f1'])}"      f"{pct(conf_overall['citation_f1'])}"      f"{pct(conf_overall['all_citations_present'])}")conf_by_kind = []for t in sorted({r["qa_type"] for r in conf_rows}):    sub = [r for r in conf_rows if r["qa_type"] == t]    s = summarise(sub, t)    conf_by_kind.append(s)    print(f"{'  ' + t:<26}{s['n']:>6}{pct(s['exact_match'])}{pct(s['f1'])}"          f"{pct(s['citation_f1'])}{pct(s['all_citations_present'])}")

In [ ]:
# The qualitative view. For a demo this matters at least as much as the metric:# a grader can see immediately whether the model named the right section.for kind in ("collision", "merged", "split", "removed", "transition"):    picks = [i for i, e in enumerate(confusion) if e["mapping_type"] == kind][:2]    for i in picks:        e, pred = confusion[i], conf_preds[i]        ok = extract_refs(e["output"]) <= extract_refs(pred)        print("=" * 78)        print(f"[{kind}]   citations correct: {'YES' if ok else 'NO'}")        print("  Q    :", e["instruction"])        print("  gold :", textwrap.shorten(e["output"], 260, placeholder=" ..."))        print("  pred :", textwrap.shorten(pred, 260, placeholder=" ..."))        print("  trap :", textwrap.shorten(e["why_confusing"], 180, placeholder=" ..."))

---## 8. Saving the modelModel weights do **not** go in the git repository. A `flan-t5-base` checkpoint isaround 1 GB, git stores every revision of it forever, and GitHub rejects filesover 100 MB — `.gitignore` in the repo excludes `*.safetensors`, `checkpoint-*/`and `outputs/` for exactly this reason.Two destinations, in order of preference:1. **Hugging Face Hub** — public, versioned and citable, which suits the "make   everything viewable" constraint and gives the paper a concrete artifact link.   Needs a token: in Colab, add one under the key icon in the left sidebar as a   secret named `HF_TOKEN` with write access.2. **Google Drive** — the fallback if no token is configured.The cell below tries the Hub, falls back to Drive, and reports which it used.

In [ ]:
SAVE_LOCAL = "/content/model-final"trainer.save_model(SAVE_LOCAL)tokenizer.save_pretrained(SAVE_LOCAL)print("saved locally to", SAVE_LOCAL)hf_token = os.environ.get("HF_TOKEN")if not hf_token:    try:        from google.colab import userdata        hf_token = userdata.get("HF_TOKEN")    except Exception:        hf_token = NoneMODEL_ARTIFACT = Noneif hf_token:    from huggingface_hub import HfApi    who = HfApi(token=hf_token).whoami()["name"]    repo_id = f"{who}/{HF_REPO_ID}"    model.push_to_hub(repo_id, token=hf_token)    tokenizer.push_to_hub(repo_id, token=hf_token)    MODEL_ARTIFACT = f"https://huggingface.co/{repo_id}"    print("pushed to", MODEL_ARTIFACT)else:    print("No HF_TOKEN found - falling back to Google Drive.")    print("To use the Hub instead: Colab left sidebar > key icon > add a secret")    print("named HF_TOKEN with a write-scoped token, then re-run this cell.")    from google.colab import drive    drive.mount("/content/drive")    dest = "/content/drive/MyDrive/legal-llm-bot/flan-t5-base-finetuned"    os.makedirs(dest, exist_ok=True)    model.save_pretrained(dest)    tokenizer.save_pretrained(dest)    MODEL_ARTIFACT = dest    print("saved to", MODEL_ARTIFACT)

---## 9. Documentation and resultsThe cell below assembles the run summary from the values actually producedabove, rather than from numbers typed in by hand, and writes it to`/content/phase2_results.json` so it can be pasted into the paper withouttranscription errors.

In [ ]:
summary = {    "model": MODEL_NAME,    "epochs_run": float(EPOCHS_RUN),    "epoch_ceiling": 25,    "final_train_loss": float(FINAL_TRAIN_LOSS),    "best_val_loss": float(BEST_VAL_LOSS),    "training_minutes": round(TRAIN_MINUTES, 1),    "precision": "bf16" if USE_BF16 else "fp32",    "effective_batch_size": BATCH_SIZE * GRAD_ACCUM,    "learning_rate": 3e-4,    "rows": {n: len(v) for n, v in splits.items()},    "test_overall": overall,    "test_by_qa_type": by_type,    "confusion_overall": conf_overall,    "confusion_by_kind": conf_by_kind,    "model_artifact": MODEL_ARTIFACT,    "github_repo": REPO_URL,}with open("/content/phase2_results.json", "w", encoding="utf-8") as fh:    json.dump(summary, fh, indent=2)print(f"Trained {EPOCHS_RUN:.0f}/25 epochs in {TRAIN_MINUTES:.1f} min "      f"({'bf16' if USE_BF16 else 'fp32'}); best val loss {BEST_VAL_LOSS:.4f}")print(f"Test  : EM{pct(overall['exact_match'])}  F1{pct(overall['f1'])}  "      f"citation F1{pct(overall['citation_f1'])}")print(f"Confus: EM{pct(conf_overall['exact_match'])}  "      f"F1{pct(conf_overall['f1'])}  "      f"citation F1{pct(conf_overall['citation_f1'])}")print("\nartifact:", MODEL_ARTIFACT)print("results written to /content/phase2_results.json")

### Reading the resultsFill this in after the run — the numbers print above and are saved to`phase2_results.json`.**What to look for, and what it means:*** **Per-type spread is the real result.** If `section_text` scores well and  `old_to_new` / `new_to_old` do not, the model has learnt to recite statute text  but not the correspondence, and the aggregate is hiding it. The citation F1  column is the one to read for those two types.* **`offence_classification` is a good sanity check.** Cognizable / bailable /  which court is a small closed vocabulary, so it should score high. If it does  not, something is wrong with training rather than with the task.* **Low exact match with high F1 is expected**, not a problem — the answers are  sentences, and EM demands the sentence back verbatim.* **The confusion set should be the weakest number here.** It is a no-retrieval  baseline on facts deliberately withheld from training. Record it: Phase 4  measures the retrieval layer against exactly this.### Issues encounteredNote anything hit during the run. The two that are anticipated:* **fp16 produces NaN losses with T5.** Known behaviour — activations overflow  the fp16 range. Handled by training in fp32, since the T4 is Turing and has no  bf16 support. Costs speed, not correctness.* **Out of memory.** If the T4 runs out, lower `BATCH_SIZE` to 4 and raise  `GRAD_ACCUM` to 4, which keeps the effective batch at 16. Failing that, drop  `MAX_TARGET_LENGTH` to 256, which truncates about 1.8% of training targets.### What Phase 3 addsThis model answers from what it memorised during fine-tuning. That is the wrongfoundation for a legal assistant on its own: it cannot cite a source, cannot beupdated when the law changes, and fails silently on anything it did not see.Phase 3 puts a FAISS retrieval layer over `retrieval_corpus.jsonl` — 2,819 chunksat one chunk per section, First Schedule entry or judgment, each carrying itsold/new counterpart inside the embedded text so that a question about "IPC 302"retrieves the BNS section that replaced it. Retrieved passages go into the`context` slot that this notebook's prompt template already reserves, so theinput format does not change. That is what lets the bot cite a source, and whatthe confusion set is really testing.Phase 4 then runs the confusion set against this fine-tuned model, theretrieval-augmented version, and a general-purpose LLM side by side.